[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Fancy Indexing &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Pick Oslo's and Tromso's rows, 1 and 3, with one index, and print the shape of the result.

In [2]:
print(readings[[1, 3]].shape)

(2, 8760)


One list of two row positions keeps both rows whole.

**2.** Print Bergen's readings at midnight on the first of every month, using `hour_of_year` and a list of positions.

In [3]:
midnights = [hour_of_year(month, 1) for month in range(1, 13)]

print(readings[0, midnights])

[-3.6 -3.1 -0.1  2.6  7.3 12.1 14.6 15.4 11.9  8.1  3.9 -0.1]


`hour_of_year` counts from midnight when no hour is given.

**3.** Print the station names from the warmest yearly mean to the coldest, using `np.argsort` with `kind="stable"`.

In [4]:
names = np.array(list(STATIONS))
means = np.nanmean(readings, axis=1)

print(names[np.argsort(means, kind="stable")[::-1]])

['Bergen' 'Oslo' 'Tromso' 'Svalbard']


`np.argsort` sorts from smallest to largest, so `[::-1]` turns the positions around. `np.nanmean` leaves Svalbard's missing hours out of its mean.

**4.** Find Oslo's three warmest hours with `np.argsort`, warmest first, and print the date and time of each with `START` and `timedelta`.

In [5]:
warm_hours = np.argsort(readings[1], kind="stable")[-3:][::-1]

print(readings[1, warm_hours])
for hour in warm_hours:
    print(START + timedelta(hours=int(hour)))

[19.3 19.2 19.2]
2025-07-17 15:00:00
2025-07-22 14:00:00
2025-07-21 15:00:00


Two of the three hours were 19.2 degrees. The stable sort keeps them in the order they had in the year, and `[::-1]` reverses that order along with the rest.

**5.** Make a grid of Svalbard and Oslo, rows 2 and 1, at noon on 1 January and noon on 1 July with `np.ix_`, and print it.

In [6]:
noon_hours = [hour_of_year(1, 1, 12), hour_of_year(7, 1, 12)]

print(readings[np.ix_([2, 1], noon_hours)])

[[-11.8   6.4]
 [ -0.7  17.5]]


The rows come out in the order of the first list, Svalbard above Oslo.

---

&#8592; **Back to:** [Fancy Indexing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/07-fancy-indexing.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
